# RAG Evaluation — University Assistant
Evaluates retrieval quality using Precision@k, Recall@k, and Hit Rate.

In [ ]:
import sys, json
sys.path.insert(0, '..')
from backend.rag.vector_store import initialize_vector_store
from backend.rag.retriever import get_retriever

initialize_vector_store()
retriever = get_retriever()
print('Vector store ready.')

## Test Queries with Ground Truth

In [ ]:
# Define evaluation set: (query, expected_keywords_in_result)
eval_set = [
    {
        'query': 'What are prerequisites for DS201?',
        'expected_keywords': ['DS201', 'CS101', 'STAT101'],
        'doc_type': 'course',
    },
    {
        'query': 'When does registration close?',
        'expected_keywords': ['registration', 'close', 'deadline'],
        'doc_type': 'calendar',
    },
    {
        'query': 'What is the tuition fee for BTech domestic students?',
        'expected_keywords': ['BTech', 'tuition', '85000'],
        'doc_type': 'fees',
    },
    {
        'query': 'What is the attendance policy?',
        'expected_keywords': ['75%', 'attendance', 'debarred'],
        'doc_type': 'policy',
    },
    {
        'query': 'What are the library timings?',
        'expected_keywords': ['library', '8:00', '10:00', 'borrow'],
        'doc_type': 'policy',
    },
    {
        'query': 'Who teaches Machine Learning ML301?',
        'expected_keywords': ['Kiran Mehta', 'ML301'],
        'doc_type': 'faculty',
    },
    {
        'query': 'What scholarships are available for PhD students?',
        'expected_keywords': ['PhD', 'scholarship', 'fellowship'],
        'doc_type': 'scholarship',
    },
    {
        'query': 'What are hostel curfew rules?',
        'expected_keywords': ['curfew', 'hostel', '10:00 PM'],
        'doc_type': 'policy',
    },
]
print(f'{len(eval_set)} evaluation queries loaded.')

In [ ]:
import pandas as pd

def precision_at_k(retrieved_docs, expected_keywords, k=5):
    relevant = 0
    for doc in retrieved_docs[:k]:
        content = doc['content'].lower()
        if any(kw.lower() in content for kw in expected_keywords):
            relevant += 1
    return relevant / k

def recall_at_k(retrieved_docs, expected_keywords, k=5):
    found_keywords = set()
    for doc in retrieved_docs[:k]:
        content = doc['content'].lower()
        for kw in expected_keywords:
            if kw.lower() in content:
                found_keywords.add(kw.lower())
    return len(found_keywords) / len(expected_keywords)

def hit_rate(retrieved_docs, expected_keywords, k=5):
    for doc in retrieved_docs[:k]:
        content = doc['content'].lower()
        if any(kw.lower() in content for kw in expected_keywords):
            return 1
    return 0

results = []
for item in eval_set:
    docs = retriever.retrieve(item['query'], top_k=5, doc_type=item.get('doc_type'))
    p = precision_at_k(docs, item['expected_keywords'])
    r = recall_at_k(docs, item['expected_keywords'])
    h = hit_rate(docs, item['expected_keywords'])
    results.append({
        'Query': item['query'][:60],
        'Precision@5': round(p, 3),
        'Recall@5': round(r, 3),
        'HitRate': h,
        'Docs Retrieved': len(docs),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nAverage Precision@5: {df['Precision@5'].mean():.3f}")
print(f"Average Recall@5: {df['Recall@5'].mean():.3f}")
print(f"Hit Rate: {df['HitRate'].mean():.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

queries_short = [q[:30]+'...' for q in df['Query']]
x = np.arange(len(queries_short))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width, df['Precision@5'], width, label='Precision@5', color='steelblue')
ax.bar(x, df['Recall@5'], width, label='Recall@5', color='darkorange')
ax.bar(x + width, df['HitRate'], width, label='Hit Rate', color='green')
ax.set_xticks(x)
ax.set_xticklabels(queries_short, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1.2)
ax.set_ylabel('Score')
ax.set_title('RAG Evaluation Metrics')
ax.legend()
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='0.5 threshold')
plt.tight_layout()
plt.savefig('rag_eval_results.png', dpi=150)
plt.show()
print('Plot saved.')